In [6]:
import pandas as pd
import os

In [7]:
city = 'Chennai'

df = pd.read_json(rf'../web scraping/{city}/car_dataset_{city.lower()}.json', lines=True)
df.head()

,url,car_name,Price,Registration Year,Insurance,Fuel Type,Seats,Kms Driven,RTO,Ownership,...,Charging Time AC,Motor Type,Charging Time (A.C),Charging Time (D.C),Charging Port,Acceleration 0-100kmph,Charging Time,Fast Charging,Regenerative Braking,Battery Type
0,https://www.cardekho.com/used-car-details/used...,BMW 6 Series,₹37.90 Lakh,2018,-,Diesel,4 Seats,"90,000 Kms",karaikal,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.cardekho.com/used-car-details/used...,Volkswagen Tiguan,₹17 Lakh,Nov 2018,-,Diesel,5 Seats,"81,000 Kms",Chennai,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.cardekho.com/used-car-details/used...,Ford Figo,₹4.10 Lakh,Jul 2015,Comprehensive,Diesel,5 Seats,"82,208 Kms",Chennai,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.cardekho.com/used-car-details/used...,Kia Seltos,₹11.50 Lakh,Jan 2019,-,Diesel,5 Seats,"80,000 Kms",Chengalpattu,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.cardekho.com/used-car-details/used...,Hyundai Grand i10 Nios,₹6.20 Lakh,2019,-,Petrol,5 Seats,"36,000 Kms",Chennai,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
PROCESSED_FILES_LOG = 'processed_files.txt'

def get_processed_files():
    if os.path.exists(PROCESSED_FILES_LOG):
        with open(PROCESSED_FILES_LOG, 'r') as f:
            return f.read().splitlines()
    return []

def mark_file_as_processed(json_filename):
    processed = get_processed_files()
    if json_filename not in processed:
        with open(PROCESSED_FILES_LOG, 'a') as f:
            f.write(json_filename + '\n')

In [9]:
# ─────────────────────────────────────────
json_file = f'car_dataset_{city.lower()}.json'

if json_file in get_processed_files():
    print(f"⚠️ '{json_file}' already processed — skipping!")
    df = pd.read_csv('dataset.csv')  
    print(f"Loaded existing dataset with {len(df)} rows")
    
else:  
    req_col = []
    with open('features.txt', 'r') as f:
        features = f.read().split('\n')
    for i in features:
        if i in df.columns:
            req_col.append(i) 
    df = df[req_col]
    df['city'] = city

    before = len(df)
    df = df.drop_duplicates()
    print(f"Duplicates removed from new data: {before - len(df)} rows")

    if os.path.exists('dataset.csv') and os.path.getsize('dataset.csv') > 0:
        existing_df = pd.read_csv('dataset.csv')
        combined_df = pd.concat([existing_df, df], ignore_index=True)

        before = len(combined_df)
        combined_df = combined_df.drop_duplicates()
        print(f"Duplicates removed from combined data: {before - len(combined_df)} rows")

        combined_df.to_csv('dataset.csv', index=False)
        print(f"Appended {len(df)} rows. Total rows: {len(combined_df)}")
    else:
        df.to_csv('dataset.csv', index=False)
        print(f"Created new dataset.csv with {len(df)} rows")

    mark_file_as_processed(json_file)
    print(f"✅ '{json_file}' marked as processed!")

Duplicates removed from new data: 688 rows
Duplicates removed from combined data: 0 rows
Appended 824 rows. Total rows: 5241
✅ 'car_dataset_chennai.json' marked as processed!
